In [ ]:
import contraqctor
import logging
from dataclasses import dataclass, field
import numpy as np
import pandas as pd

from contraqctor.contract.harp import HarpDevice
from contraqctor.qc.harp import HarpDeviceTestSuite
from contraqctor.qc.base import Runner
from aind_behavior_vr_foraging.data_contract import dataset as vr_foraging_dataset

In [ ]:
from aind_vr_foraging_analysis.utils.parsing import data_access
date_string = "2026-06-01" # YYYY-MM-DD
tests_results = []
mouse_list = ['846440', '846441', '846439', '849495', '863680', '863700', '863703', '863704', '863696', '863698', '864847', '841299', '841312']
# This section will look at all the session paths that fulfill the condition
for mouse in mouse_list:
    session_paths = data_access.find_sessions_relative_to_date(
        mouse=mouse,
        date_string=date_string,
        when='on_or_after'
    )

    # Iterate over the session paths and load the data
    for session_path in session_paths:
        # print(f"Loading {session_path.name}...")
        # try:
        dataset = vr_foraging_dataset(session_path)
        print(dataset['Behavior'].data[6]['WhoAmI'].data)
        suite = HarpDeviceTestSuite(dataset['Behavior'].data[6].load()).test_registers_are_monotonicity()
        print(suite.status)
        lists = {'status': suite.status, 'session_path': session_path, 'device': dataset['Behavior'].data[6]['WhoAmI'].data['WhoAmI'].unique()[0]}
        tests_results.append(lists)

results = pd.DataFrame(tests_results)

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from harp import read

DATA_ROOT = Path(r"Z:/stage/vr-foraging/data/841312/841312_2026-06-16T191514Z")
COUNTER_REG = "WhiteRabbit_33.bin"


def get_rig_name(session_dir: Path) -> str:
    rig_json = session_dir / "behavior" / "Logs" / "rig_input.json"
    if not rig_json.exists():
        return "unknown"
    return json.loads(rig_json.read_text()).get("rig_name", "unknown")


def check_session(session_dir: Path) -> dict | None:
    f = session_dir / "behavior" / "ClockGenerator.harp" / COUNTER_REG
    print(f"Checking {f}...")
    if not f.exists():
        print(f"File {f} does not exist. Skipping.")
        return None
    data = read(f)
    ts = data.index.to_numpy()
    vals = data.iloc[:, 0].to_numpy()
    ts_diffs = np.diff(ts)
    val_diffs = np.diff(vals)
    ts_jumps = np.where(ts_diffs < 0)[0]
    val_jumps = np.where(val_diffs < 0)[0]
    return {
        "session": session_dir.name,
        "rig": get_rig_name(session_dir),
        "n_rows": len(ts),
        "ts_backward_jumps": len(ts_jumps),
        "val_backward_jumps": len(val_jumps),
        "ts_jump_magnitude_s": float(ts_diffs[ts_jumps].min()) if len(ts_jumps) else 0.0,
    }
    
sessions = sorted(p for p in DATA_ROOT.iterdir() if p.is_dir() and p.name != "processed")
results = [r for s in sessions if (r := check_session(s)) is not None]
df = pd.DataFrame(results).set_index("session")

rigs = sorted(df["rig"].unique())
fig, axes = plt.subplots(2, len(rigs), figsize=(7 * len(rigs), 8), sharey="row")
if len(rigs) == 1:
    axes = axes[:, np.newaxis]

for col, rig in enumerate(rigs):
    sub = df[df["rig"] == rig]
    labels = [s[-19:] for s in sub.index]
    x = np.arange(len(sub))

    axes[0, col].bar(x, sub["ts_backward_jumps"], color="steelblue")
    axes[0, col].set_title(f"{rig}\nBackward timestamp jumps")
    axes[0, col].set_ylabel("Count")

    axes[1, col].bar(x, sub["val_backward_jumps"], color="tomato")
    axes[1, col].set_title("Backward counter-value jumps")
    axes[1, col].set_ylabel("Count")
    axes[1, col].set_xticks(x)
    axes[1, col].set_xticklabels(labels, rotation=45, ha="right", fontsize=8)

fig.suptitle("WhiteRabbit Counter (reg 33) — non-monotonicity per session", fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
df[df["rig"] == rig].groupby("session")["ts_backward_jumps"].nunique()

In [ ]:
from aind_vr_foraging_analysis.utils.parsing import data_access
tests_results = []
mouse_list = ['863680', '863700','841299', '841312', '841314', '841300', '841301', '841302',  '841306','841310']
# This section will look at all the session paths that fulfill the condition
for mouse in mouse_list:
    DATA_ROOT = Path(rf"Z:/stage/vr-foraging/data/{mouse}")
    sessions = sorted(p for p in DATA_ROOT.iterdir() if p.is_dir() and p.name != "processed")
    results = [r for s in sessions if (r := check_session(s)) is not None]
    df = pd.DataFrame(results).set_index("session")
    print(len(df), "sessions found for mouse", mouse)
    rigs = sorted(df["rig"].unique())
    fig, axes = plt.subplots(2, len(rigs), figsize=(7 * len(rigs), 8), sharey="row")
    if len(rigs) == 1:
        axes = axes[:, np.newaxis]

    for col, rig in enumerate(rigs):
        sub = df[df["rig"] == rig]
        print(sub)
        labels = [s[-19:] for s in sub.index]
        x = np.arange(len(sub))

        axes[0, col].bar(x, sub["ts_backward_jumps"], color="steelblue")
        axes[0, col].set_title(f"{rig}\nBackward timestamp jumps")
        axes[0, col].set_ylabel("Count")

        axes[1, col].bar(x, sub["val_backward_jumps"], color="tomato")
        axes[1, col].set_title("Backward counter-value jumps")
        axes[1, col].set_ylabel("Count")
        axes[1, col].set_xticks(x)
        axes[1, col].set_xticklabels(labels, rotation=45, ha="right", fontsize=8)

    fig.suptitle("WhiteRabbit Counter (reg 33) — non-monotonicity per session", fontsize=13)
    fig.tight_layout()
    plt.show()

In [ ]:
@dataclass
class ProcessedLickometer:
    onsets: np.ndarray
    frequency: pd.DataFrame
    
def process_lickometer(
    data, *, refractory_period_s: float = 0.05, dt_resample: float = 0.1
) -> ProcessedLickometer | None:
    from contraqctor.qc.harp import lickety_split

    data['harp_lickometer'].streams.LickState.load_from_file()
    data['harp_lickometer'].streams.TimestampSeconds.load_from_file()
    lickometer = data['harp_lickometer'].streams.LickState.data.copy()
    lickometer = lickometer[lickometer["MessageType"] == "EVENT"]["Channel0"]
    lick_onsets = lickometer[(lickometer) & (~lickometer.shift(1, fill_value=False))].index
    if len(lick_onsets) == 0:
        logging.warning("No lick onsets found in lickometer data.")
        return None

    suite = lickety_split.HarpLicketySplitTestSuite(
        data['harp_lickometer'].streams, lick_refractory_period=refractory_period_s
    )
    test_minimum_lick_rate = suite.test_minimum_lick_rate()
    if test_minimum_lick_rate.status != contraqctor.qc.Status.PASSED:
        logging.warning(
            f"Lickometer data quality test failed: Minimum Lick Rate Test - {test_minimum_lick_rate.result}"
        )
        return None

    keep = np.ones(len(lick_onsets), dtype=bool)
    keep[1:] = np.diff(lick_onsets) >= refractory_period_s
    kept = lick_onsets[keep]

    t_start = lickometer.index.values[0]
    t_end = lickometer.index.values[-1]

    bin_edges = np.arange(t_start, t_end + dt_resample, dt_resample)
    counts, _ = np.histogram(kept, bins=bin_edges)

    frequency_hz = counts / dt_resample

    bin_centers = bin_edges[:-1] + dt_resample / 2

    frequency = pd.Series(
        frequency_hz,
        index=pd.Index(bin_centers, name="Seconds"),
        name="frequency",
    )

    return ProcessedLickometer(
        onsets=kept,
        frequency=frequency,
    )